# Leave-one-subject-out (LOSO) evaluation
Train on 9 subjects of THINGS-EEG2 and test zero-shot retrieval on the held-out subject. Also measures Procrustes calibration with 25, 50 and 100 labelled trials of the new subject.

Preprocessed data, features and per-epoch checkpoints are cached on Google Drive, so an interrupted run resumes where it stopped.

## Setup

In [ ]:
# Setup: mount Drive, paths, settings
import os, sys, gc, json, math, time, random, shutil, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")   # results and checkpoints
DCACHE = f"{DRIVE}/cache"                                                     # preprocessed EEG and image features
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")                      # raw downloads
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")                        # local copy of the cache
for d in [DRIVE, DCACHE, ROOT, LOCAL]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "openai"
LR, WD = 3e-4, 1e-4

free_gb = shutil.disk_usage(DRIVE).free / 1e9
have = sum(os.path.exists(f"{DCACHE}/{s}_train_z16.npy") for s in ALL_SUBJECTS)
print(f"Drive free: {free_gb:.1f} GB | subjects already saved in Drive: {have}/10")
if free_gb < 0.55 * (10 - have) + 0.2:
    print("warning: low Drive space (about 0.53 GB per subject is needed)")

import numpy as np
for pkg, pip_name in [("open_clip", "open_clip_torch"), ("huggingface_hub", "huggingface_hub"), ("pandas", "pandas")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def drive_ok(path, shape0=None):
    try:
        a = np.load(path, mmap_mode="r"); return shape0 is None or a.shape[0] == shape0
    except Exception:
        return False


## Preprocessing

In [ ]:
# Preprocessing: average repetitions, per-subject channel z-score, store as float16.
# Each split runs in a separate process to limit peak memory.
PREP_SCRIPT = r'''
import numpy as np, os, sys, time
eeg_dir, out_dir, sub, split = sys.argv[1:5]
src = f"{eeg_dir}/{sub}/preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
t0 = time.time()
obj = np.load(src, allow_pickle=True)
d = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
arr = d["preprocessed_eeg_data"]
avg = np.empty((arr.shape[0],) + arr.shape[2:], dtype=np.float32)
for i in range(0, arr.shape[0], 500):
    avg[i:i + 500] = arr[i:i + 500].mean(axis=1)
del d, arr, obj
m = avg.mean(axis=(0, 2), keepdims=True); s = avg.std(axis=(0, 2), keepdims=True) + 1e-6
out = np.empty(avg.shape, dtype=np.float16)
for i in range(0, avg.shape[0], 500):
    out[i:i + 500] = (avg[i:i + 500] - m) / s
tmp = f"{out_dir}/{sub}_{split}_z16.tmp.npy"
np.save(tmp, out); os.replace(tmp, f"{out_dir}/{sub}_{split}_z16.npy")
print(f"  {sub}/{split}: {out.shape} saved in {time.time()-t0:.0f}s", flush=True)
'''
EEG_DIR = f"{ROOT}/Preprocessed_data_250Hz"
N_ROWS = {"train": 16540, "test": 200}
for sub in ALL_SUBJECTS:
    for split in ["train", "test"]:
        if drive_ok(f"{DCACHE}/{sub}_{split}_z16.npy", N_ROWS[split]): continue
        fname = f"preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
        raw = f"{EEG_DIR}/{sub}/{fname}"
        if not os.path.exists(raw):
            from huggingface_hub import hf_hub_download
            hf_hub_download(repo_id="LidongYang/EEG_Image_decode", repo_type="dataset", local_dir=ROOT,
                            filename=f"Preprocessed_data_250Hz/{sub}/{fname}")
        r = subprocess.run([sys.executable, "-c", PREP_SCRIPT, EEG_DIR, DCACHE, sub, split], capture_output=True, text=True)
        print(r.stdout.strip())
        if r.returncode != 0:
            print(r.stderr[-1500:])
            raise RuntimeError(f"preparing {sub}/{split} failed (message above)")
        if os.path.exists(raw): os.remove(raw)
    print(f"{sub} done")



## Image features

In [ ]:
# Image features (frozen CLIP), cached
feat_path = f"{DCACHE}/feats_{CLIP_ARCH}_{CLIP_PRETRAINED}.pt"
try:
    FEAT = torch.load(feat_path); print("features loaded")
except Exception:
    import tarfile, open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz",
                                   repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRETRAINED)
    clip_model = clip_model.to(device).eval()
    class ImgDS(Dataset):
        def __init__(self, paths): self.paths = paths
        def __len__(self): return len(self.paths)
        def __getitem__(self, i): return preprocess(Image.open(self.paths[i]).convert("RGB"))
    @torch.no_grad()
    def encode_images(paths):
        out = []
        for x in DataLoader(ImgDS(paths), batch_size=128, num_workers=2):
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out.append(F.normalize(clip_model.encode_image(x.to(device)).float(), dim=-1).cpu())
        return torch.cat(out)
    t0 = time.time()
    FEAT = {"train_img": encode_images([idx_train[f] for f in train_files]),
            "test_img": encode_images([idx_test[f] for f in test_files]),
            "n_concepts": len(set(meta["train_img_concepts"]))}
    torch.save(FEAT, feat_path + ".tmp"); os.replace(feat_path + ".tmp", feat_path)
    del clip_model; torch.cuda.empty_cache() if device == "cuda" else None
    print(f"features computed in {time.time()-t0:.0f}s")
N_TR = FEAT["train_img"].shape[0]; D_EMB = FEAT["train_img"].shape[1]
print("train images", N_TR, "| test images", FEAT["test_img"].shape[0], "| concepts", FEAT["n_concepts"])


## LOSO training and evaluation

In [ ]:
# Leave-one-subject-out: train on 9 subjects, test on the held-out subject.
# A checkpoint is written after each epoch so interrupted runs can resume.
LOSO_FOLDS = ["sub-01", "sub-05", "sub-08"]      # or ALL_SUBJECTS
LOSO_EPOCHS, LOSO_PATIENCE, LOSO_SEED, LOSO_BATCH = 20, 5, 0, 256

class EEGEncoder(nn.Module):                     # same encoder as the baseline
    def __init__(self, n_ch, n_t, out_dim, k=40, drop=0.5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, k, (1, 25)), nn.AvgPool2d((1, 51), (1, 5)), nn.BatchNorm2d(k), nn.ELU(),
            nn.Conv2d(k, k, (n_ch, 1)), nn.BatchNorm2d(k), nn.ELU(), nn.Dropout(drop))
        with torch.no_grad(): flat = self.backbone(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(flat, out_dim))
        self.res = nn.Sequential(nn.GELU(), nn.Linear(out_dim, out_dim), nn.Dropout(drop))
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        h = self.proj(self.backbone(x.unsqueeze(1)))
        return F.normalize(self.norm(h + self.res(h)), dim=-1)

@torch.no_grad()
def embed(model, X, bs=512):
    model.eval(); return torch.cat([model(X[i:i + bs].float()) for i in range(0, len(X), bs)])

@torch.no_grad()
def retrieval(z, targets, ks=(1, 5)):
    sim = z @ targets.T
    rank = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {f"top{k}": (rank < k).float().mean().item() * 100 for k in ks}

def procrustes(X, Y, rho=0.0):
    M = X.T @ Y
    if rho > 0: M = M + rho * torch.linalg.matrix_norm(M, ord=2) * torch.eye(M.shape[0], device=M.device)
    U, _, Vh = torch.linalg.svd(M)
    return U @ Vh

def local_copy(name):                            # copy cache to local disk
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

_, N_CH, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
PER = N_TR // FEAT["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)    # one image per training concept
VAL_IDX = torch.from_numpy(np.where(IS_VAL)[0]).to(device)
IMG_tr, IMG_te = FEAT["train_img"].to(device), FEAT["test_img"].to(device)
E = torch.arange(100, 200, device=device)        # evaluation half (100-way)
C = torch.arange(0, 100, device=device)          # calibration half

def run_fold(test_sub):
    set_seed(LOSO_SEED); t0 = time.time()
    srcs = [s for s in ALL_SUBJECTS if s != test_sub]
    X = torch.empty((len(srcs) * N_TR, N_CH, N_T), dtype=torch.float16, device=device)   # about 4.7 GB
    for k, s in enumerate(srcs):
        X[k * N_TR:(k + 1) * N_TR] = torch.from_numpy(np.load(local_copy(f"{s}_train_z16.npy"))).to(device)
    print(f"  {test_sub}: data loaded in {(time.time()-t0)/60:.1f} min", flush=True)
    img_of = torch.arange(len(X), device=device) % N_TR
    tr_idx = torch.where(~torch.from_numpy(np.tile(IS_VAL, len(srcs))).to(device))[0]

    model = EEGEncoder(N_CH, N_T, D_EMB).to(device)
    logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=device))
    opt = torch.optim.AdamW(list(model.parameters()) + [logit_scale], lr=LR, weight_decay=WD)
    st = {"ep": 0, "best": -1.0, "best_ep": 0, "bad": 0, "best_state": None}
    ckpt = f"{DRIVE}/ckpt_{test_sub}.pt"
    if os.path.exists(ckpt):
        try:
            c = torch.load(ckpt, map_location=device, weights_only=False)
            model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"])
            with torch.no_grad(): logit_scale.copy_(c["logit_scale"])
            st = c["st"]; torch.set_rng_state(c["rng"].cpu())
            if device == "cuda" and c.get("cuda_rng") is not None: torch.cuda.set_rng_state(c["cuda_rng"].cpu())
            print(f"  {test_sub}: resumed at epoch {st['ep']}", flush=True)
        except Exception as e:
            print(f"  {test_sub}: checkpoint unreadable ({e}), restarting", flush=True)

    while st["ep"] < LOSO_EPOCHS and st["bad"] < LOSO_PATIENCE:
        model.train(); perm = tr_idx[torch.randperm(len(tr_idx), device=device)]
        for i in range(0, len(perm), LOSO_BATCH):
            b = perm[i:i + LOSO_BATCH]; ib = img_of[b]
            z = model(X[b].float()); s_ = logit_scale.exp().clamp(max=100)
            logits = s_ * z @ IMG_tr[ib].T
            same = (ib[:, None] == ib[None, :]).float()     # same image across subjects is a positive
            tgt = same / same.sum(1, keepdim=True)
            loss = (torch.sum(-tgt * F.log_softmax(logits, 1), 1).mean() + torch.sum(-tgt * F.log_softmax(logits.T, 1), 1).mean()) / 2
            opt.zero_grad(); loss.backward(); opt.step()
        va = float(np.mean([retrieval(embed(model, X[k * N_TR + VAL_IDX]), IMG_tr[VAL_IDX])["top5"] for k in range(len(srcs))]))
        st["ep"] += 1
        if va > st["best"]:
            st.update(best=va, best_ep=st["ep"], bad=0, best_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        else:
            st["bad"] += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "logit_scale": logit_scale.detach(), "st": st,
                    "rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state() if device == "cuda" else None}, ckpt + ".tmp")
        os.replace(ckpt + ".tmp", ckpt)
        print(f"  {test_sub} ep {st['ep']:2d} | val top5 {va:.2f} | {(time.time()-t0)/60:.1f} min", flush=True)

    model.load_state_dict(st["best_state"]); del X; gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None
    te = torch.from_numpy(np.load(local_copy(f"{test_sub}_test_z16.npy"))).float().to(device)
    z = embed(model, te)
    r = {"test_subject": test_sub, "best_epoch": st["best_ep"], "val_top5": st["best"],
         "loso_top1_200": retrieval(z, IMG_te)["top1"], "loso_top5_200": retrieval(z, IMG_te)["top5"],
         "loso_top1_100": retrieval(z[E], IMG_te[E])["top1"]}
    for k in [25, 50, 100]:                       # Procrustes calibration with k labelled trials
        R = procrustes(z[C[:k]], IMG_te[C[:k]], rho=0.5)
        r[f"procrustes_k{k}_top1_100"] = retrieval(F.normalize(z[E] @ R, dim=-1), IMG_te[E])["top1"]
    return r

for sub in LOSO_FOLDS:
    path = f"{DRIVE}/loso_{sub}.json"
    if os.path.exists(path): print(f"{sub}: done"); continue
    r = run_fold(sub); json.dump(r, open(path, "w"))
    if os.path.exists(f"{DRIVE}/ckpt_{sub}.pt"): os.remove(f"{DRIVE}/ckpt_{sub}.pt")
    print(f"{sub}: LOSO 200-way top1 {r['loso_top1_200']:.1f}%  top5 {r['loso_top5_200']:.1f}% | "
          f"100-way zero-shot {r['loso_top1_100']:.0f}% -> Procrustes k100 {r['procrustes_k100_top1_100']:.0f}%", flush=True)

rows = [json.load(open(f"{DRIVE}/loso_{s}.json")) for s in ALL_SUBJECTS if os.path.exists(f"{DRIVE}/loso_{s}.json")]
loso = pd.DataFrame(rows); loso.to_csv(f"{DRIVE}/results_loso.csv", index=False)
print(loso.round(1).to_string(index=False))
print(f"mean over {len(loso)} folds: top1 {loso.loso_top1_200.mean():.1f}%, top5 {loso.loso_top5_200.mean():.1f}%")
